In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from visualize_path import predictions2rel, rel2abs, kitti2abs


def abs2rel(abs_poses):
    inv_transform_matrices = abs_poses

    rot_matrices = np.linalg.inv(inv_transform_matrices[:,:,:3])
    tr_vectors = -rot_matrices @ inv_transform_matrices[:,:,-1:]

    transform_matrices = np.concatenate([rot_matrices, tr_vectors], axis=-1)

    first_inv_transform = inv_transform_matrices[0]
    final_poses = first_inv_transform[:,:3] @ transform_matrices
    final_poses[:,:,-1:] += first_inv_transform[:,-1:]
    
    return final_poses


def sliding_window_true_scales(true_translations, pred_translations, radius=2):
    true_scales = []

    for i in range(true_translations.shape[0]):
        window = np.s_[max(0, i - radius): min(i + radius + 1, true_translations.shape[0])]
        true_scale = np.sum(true_translations[window] * pred_translations[window]) / np.sum(np.square(pred_translations[window]))
        true_scales.append(true_scale)

    return np.array(true_scales)

In [ ]:
sequence = "kitti_odometry_09"

pred_abs_poses = rel2abs(predictions2rel(np.load(f"path_examples/{sequence}.npy")))
true_abs_poses = kitti2abs(f"path_examples/{sequence}.txt")

pred_scales = np.load(f"path_examples/{sequence}_scales.npy", allow_pickle=True).item()
pred_scales = 1 / np.array([item[1] for item in sorted(pred_scales.items(), key=lambda item: int(item[0]))])

assert pred_abs_poses.shape == true_abs_poses.shape
assert true_abs_poses.shape[0] == pred_scales.shape[0]

true_translations = true_abs_poses[:, :, -1]
pred_translations = pred_abs_poses[:, :, -1]

true_scales = sliding_window_true_scales(true_translations, pred_translations)

print(np.median(pred_scales) / np.median(true_scales))

pred_scales /= 5  # magick

In [ ]:
plt.rcParams.update({'font.size': 20})
plt.figure(figsize=(12, 8))

plt.plot(true_scales, label="true scale (knowing GT)")
plt.plot(pred_scales, label="pred scale")
plt.axhline(y=np.median(pred_scales), label="pred scale (median)")

plt.xlabel("frame")
plt.ylabel("scale")
plt.gca().set_ylim([0, 30])
plt.legend()
plt.show()

In [ ]:
np.save(f"path_examples/{sequence}_5x_scales.npy", np.median(pred_scales) * np.ones_like(true_scales))
np.save(f"path_examples/{sequence}_true_scales.npy", true_scales)